## browser-use 익히기
- AI가 직접 브라우저를 조작해주는 오픈소스 라이브러리입니다.
- GitHub 스타가 10만 개(103k)를 넘은 인기 프로젝트입니다.
- 저장소 : https://github.com/browser-use/browser-use

### browser-use가 하는 일
쉽게 말하면, AI가 사이트에 들어가서 사람이 하듯이 직접 조작합니다.
- 검색, 클릭, 폼 입력, 로그인 같은 작업을 AI가 알아서 수행
- 예를 들어 이런 흐름이 가능합니다.

1. 내가 만들고 싶은 기능을 AI한테 말함
2. AI가 코드를 짬
3. browser-use가 브라우저에서 실제 화면을 열어봄
   - 버튼이 눌리는지
   - 로그인이 되는지
   - 폼이 제대로 작동하는지
   - 에러가 뜨는지 확인함
4. 문제가 있으면 다시 코드 수정까지 해줌

즉, **코딩 → 실제 브라우저에서 테스트 → 수정**의 루프를 AI가 스스로 돌릴 수 있습니다.

### 1. 설치하기
- Python **3.11 이상**이 필요합니다.
- 설치하면 Playwright와 Chromium(브라우저)이 함께 설치됩니다.

In [ ]:
# 주피터에서 설치할 때는 앞에 !를 붙입니다.
!pip install browser-use

# 크로미움 브라우저 설치 (최초 1회)
!playwright install chromium

### 2. API 키 설정
- browser-use는 LLM(대형 언어 모델)에게 "다음에 무엇을 클릭할지"를 물어보며 동작합니다.
- 그래서 사용할 LLM의 API 키가 필요합니다. 아래 중 하나만 있으면 됩니다.

| 사용할 모델 | 환경변수 |
|---|---|
| Browser Use 자체 모델 | `BROWSER_USE_API_KEY` |
| OpenAI (GPT) | `OPENAI_API_KEY` |
| Anthropic (Claude) | `ANTHROPIC_API_KEY` |
| Google (Gemini) | `GOOGLE_API_KEY` |

- 실무에서는 `.env` 파일에 키를 저장하고 `python-dotenv`로 불러오는 방식을 많이 씁니다.
- **주의 : API 키를 코드에 직접 적어서 GitHub에 올리면 안 됩니다!**

In [ ]:
import os

# 방법 1 : 코드에서 직접 환경변수로 지정 (연습용 - 커밋하지 말 것!)
# os.environ['ANTHROPIC_API_KEY'] = '여기에-키-입력'

# 방법 2 : .env 파일 사용 (권장)
# !pip install python-dotenv
# from dotenv import load_dotenv
# load_dotenv()

### 3. 기본 사용법
- 핵심은 `Agent` 클래스 하나입니다.
- `task`에 **자연어로 시킬 일**을 적고, `llm`에 사용할 모델을 넣으면 끝입니다.
- 실행하면 실제 크로미움 브라우저가 열리면서 AI가 마우스/키보드를 조작하는 것을 볼 수 있습니다.

In [ ]:
from browser_use import Agent, ChatBrowserUse

agent = Agent(
    # 시킬 일을 자연어로 적는다
    task="browser-use 깃허브 저장소의 스타 개수를 찾아줘",
    llm=ChatBrowserUse(),
)

# 주피터 노트북에서는 await로 바로 실행할 수 있다.
history = await agent.run()

> 참고 : 일반 `.py` 스크립트에서는 `await`를 바로 쓸 수 없어서 `asyncio`로 감싸줍니다.
```python
import asyncio
from browser_use import Agent, ChatBrowserUse

async def main():
    agent = Agent(
        task="browser-use 깃허브 저장소의 스타 개수를 찾아줘",
        llm=ChatBrowserUse(),
    )
    await agent.run()

asyncio.run(main())
```

### 4. 다른 LLM 사용하기
- OpenAI, Claude, Gemini 등 원하는 모델을 골라서 쓸 수 있습니다.

In [ ]:
# OpenAI GPT 사용
from browser_use import Agent, ChatOpenAI

agent = Agent(
    task="네이버에서 오늘 서울 날씨를 검색해서 알려줘",
    llm=ChatOpenAI(model="gpt-5.1"),
)
history = await agent.run()

In [ ]:
# Anthropic Claude 사용
from browser_use import Agent, ChatAnthropic

agent = Agent(
    task="구글에서 파이썬 pandas 공식 문서를 찾아서 주소를 알려줘",
    llm=ChatAnthropic(model="claude-sonnet-4-5"),
)
history = await agent.run()

### 5. 활용 예시 : 내가 만든 웹사이트 테스트하기
- browser-use의 강력한 활용법 중 하나가 **내가 만든 기능을 AI가 대신 테스트**해주는 것입니다.
- 사람이 일일이 클릭해보지 않아도, AI가 브라우저를 열어 직접 확인합니다.

In [ ]:
from browser_use import Agent, ChatBrowserUse

agent = Agent(
    task=(
        "http://localhost:8000 에 접속해서 다음을 확인해줘. "
        "1) 로그인 버튼이 눌리는지 "
        "2) 아이디 'test', 비밀번호 '1234'로 로그인이 되는지 "
        "3) 회원가입 폼에 값을 입력하면 제대로 동작하는지 "
        "4) 에러가 뜨는 곳이 있으면 어떤 에러인지 정리해줘"
    ),
    llm=ChatBrowserUse(),
)
history = await agent.run()

# 결과 요약 보기
print(history.final_result())

### 6. 정리
- `pip install browser-use` + `playwright install chromium` 으로 설치
- `Agent(task="자연어 지시", llm=모델)` → `await agent.run()` 이 전부
- 할 수 있는 것
  - 웹 검색, 정보 수집 자동화
  - 쇼핑, 예약 같은 반복 작업 자동화
  - **내가 만든 웹 기능의 자동 테스트** (버튼, 로그인, 폼, 에러 확인)
- 더 알아보기
  - 공식 문서 : https://docs.browser-use.com
  - 예제 모음 : https://github.com/browser-use/browser-use/tree/main/examples

### `Quiz`
1. browser-use로 좋아하는 뉴스 사이트에 들어가 오늘의 헤드라인 3개를 가져오는 Agent를 만들어보세요.
2. `task`를 어떻게 쓰느냐에 따라 결과가 달라집니다. 같은 작업을 더 구체적인 지시로 바꿔서 결과를 비교해보세요.